In [3]:
#  Step 1: Import Libraries
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

#  Step 2: Data Loading
data = pd.read_csv("cancer_variant_prioritization_dataset.csv")

#  Step 3: Data Exploration
print("Columns:\n", data.columns)
print("\nInfo:\n")
data.info()

#  Step 4: Handle Missing Values
# Numeric
data.fillna(data.select_dtypes(include=np.number).mean(), inplace=True)

# Categorical
for col in data.select_dtypes(include='object').columns:
    data[col] = data[col].fillna(data[col].mode()[0])

#  Step 5: Encode Target
label_encoder = LabelEncoder()
data['Clinical_Significance'] = label_encoder.fit_transform(data['Clinical_Significance'])

#  Step 6: Feature Selection
X = data.drop('Clinical_Significance', axis=1)
y = data['Clinical_Significance']

#  Step 7: Identify Column Types
numeric_features = X.select_dtypes(include=np.number).columns
categorical_features = X.select_dtypes(include='object').columns

#  Step 8: Preprocessing
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

#  Step 9: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#  Step 10: Model (High Accuracy)
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

#  Step 11: Pipeline
pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', model)
])

#  Step 12: Training
pipeline.fit(X_train, y_train)

#  Step 13: Prediction
y_pred = pipeline.predict(X_test)

#  Step 14: Evaluation

# Accuracy
accuracy = accuracy_score(y_test, y_pred)*100

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# Precision, Recall, F1-score
report = classification_report(y_test, y_pred)

print("\n Model Evaluation")
print("Accuracy:", accuracy)

print("\nConfusion Matrix:\n", cm)

print("\nClassification Report:\n", report)

#  Step 15: Feature Importance (Important for biomedical)
feature_names = pipeline.named_steps['preprocessing'].get_feature_names_out()
importances = pipeline.named_steps['model'].feature_importances_

feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\nTop Important Features:\n", feature_importance.head(10))

#  Step 16: Prediction on New Data
new_data = X.iloc[0:1]
prediction = pipeline.predict(new_data)

# Convert back to original label
predicted_label = label_encoder.inverse_transform(prediction)

print("\nPredicted Class:", predicted_label[0])

#  Step 17: Save Model
pickle.dump(pipeline, open("cancer_classification_model.pkl", "wb"))

print("\nModel saved successfully!")

Columns:
 Index(['Chromosome', 'Position', 'Gene', 'Variant_Type', 'Population_AF',
       'CADD_Score', 'SIFT', 'PolyPhen', 'Clinical_Significance'],
      dtype='object')

Info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Chromosome             10000 non-null  int64  
 1   Position               10000 non-null  int64  
 2   Gene                   10000 non-null  object 
 3   Variant_Type           10000 non-null  object 
 4   Population_AF          10000 non-null  float64
 5   CADD_Score             10000 non-null  float64
 6   SIFT                   10000 non-null  object 
 7   PolyPhen               10000 non-null  object 
 8   Clinical_Significance  10000 non-null  object 
dtypes: float64(2), int64(2), object(5)
memory usage: 703.3+ KB

 Model Evaluation
Accuracy: 25.45

Confusion Matrix:
 [[116 158  81 139]
 [